# H3 user-study preparation — standalone editable copy (real data only)

This is a standalone copy of `notebooks/05_user_study_prep.ipynb`. You can run it from
anywhere (including your GPU venv); it auto-detects the real repository
(`EACL DEMO 2027-final/belief_debate_analyzer`) and writes all outputs **there**, leaving
nothing fabricated behind.

What it does (nothing is generated, estimated, or backfilled at any step):

1. Loads the real, already-computed system output for the 150 human-human DebateGPT transcripts.
2. Selects the 6 real study transcripts and derives the real answer key **from the system's own output**.
3. Builds the counterbalanced workbench / raw-transcript assignment for the 3 participants (TS, GS, RK).
4. Writes the three real participant packets (`user_study_TS/GS/RK.csv`), the researcher-only
   `answer_key_master.csv` + `assignment_master.csv`, and the single fillable `results_fillable.csv`.
5. **AFTER** TS, GS and RK return their real answers, the final fail-closed cell scores them against
   the answer key and writes `data/user_study/results.csv` — the exact file notebook 03 needs to unblock H3.

Notes for running on your machine:

- The repo's canonical path on the GPU machine is `/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer`
  (this is the default this notebook tries first, matching the README). The Windows copy under
  `EACL DEMO 2027-final/belief_debate_analyzer` is tried next; partial/stale mirror folders without a
  demo site or notebooks are never picked intentionally.
- The **workbench condition needs no GPU and no HF_TOKEN**: open `demo_site/index.html` (static,
  precomputed) or run `streamlit run app.py` → "Browse a real DebateGPT example". GPU + token are only
  for the separate live-generation mode, which this study does not use.
- If auto-detection ever picks the wrong folder, set the repo path explicitly in the first code cell:
  `os.environ["EACL_REPO_ROOT"] = "/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer"`

In [1]:
import json
import os
import sys
from pathlib import Path

import pandas as pd

# Known good checkouts, tried in order before any cwd-based auto-detection.
# (1) canonical GPU-machine path from the README; (2) Windows -final working copy.
DEFAULT_REPO_ROOTS = [
    Path("/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer"),
    Path("D:/EACL SYSTEM DEMO/EACL DEMO 2027/EACL DEMO 2027-final/belief_debate_analyzer"),
]

# Optional hard override (recommended if you move this notebook around):
# os.environ["EACL_REPO_ROOT"] = "/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer"


def find_repo_root() -> Path:
    """Locate the real repo: explicit override first, then the known-good paths,
    then cwd-walked candidates. A candidate only counts when it has src/ AND the
    real system-output file, so partial mirror copies cannot be picked silently."""
    marker = Path("data") / "processed" / "debategpt_human_human_system_output.json"
    override = os.environ.get("EACL_REPO_ROOT", "").strip()
    start = Path.cwd().resolve()
    candidates: list[Path] = []
    if override:
        candidates.append(Path(override).expanduser().resolve())
    candidates.extend(p.expanduser().resolve() for p in DEFAULT_REPO_ROOTS)
    for base in [start, *start.parents]:
        candidates.append(base)
        candidates.append(base / "belief_debate_analyzer")
        candidates.append(base / "EACL DEMO 2027-final" / "belief_debate_analyzer")

    seen: set[Path] = set()
    unique = [c for c in candidates if not (c in seen or seen.add(c))]
    with_marker = [c for c in unique if (c / "src").is_dir() and (c / marker).is_file()]
    if with_marker:
        return with_marker[0]
    with_src = [c for c in unique if (c / "src").is_dir()]
    if with_src:
        print(f"WARNING: falling back to {with_src[0]} which is missing {marker}")
        return with_src[0]
    raise FileNotFoundError(
        "Could not locate the belief_debate_analyzer repo from "
        f"{start}. Set os.environ['EACL_REPO_ROOT'] to the repo path and re-run."
    )


root = find_repo_root()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

system_output_path = root / "data" / "processed" / "debategpt_human_human_system_output.json"
if not system_output_path.is_file():
    raise FileNotFoundError(f"required real system output is missing: {system_output_path}")

results = json.loads(system_output_path.read_text(encoding="utf-8"))
by_id = {r["transcript_id"]: r for r in results}
print({"status": "loaded", "transcripts": len(results), "root": str(root)})


{'status': 'loaded', 'transcripts': 150, 'root': '/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer'}


## Select real transcripts and derive real answer keys

Selection criteria (applied over all 150 real transcripts, not cherry-picked after seeing answers):
- Transcripts containing a real `strategic_persuasion` turn (there are exactly 3 in the whole corpus).
- Transcripts where exactly one participant has zero `evidence_adoption` turns and the other has at least one (a clean, unambiguous Q2 answer) and at least one `echo` turn is present (so Q3 is meaningfully "yes").

Answer-key rules (fixed before selection, applied identically to every transcript):
- **Q1**: the participant with a `strategic_persuasion` label anywhere in their turns; `"neither"` if no participant has one.
- **Q2**: the participant with zero `evidence_adoption` turns, if the other participant has at least one; `"ambiguous"` if both or neither have any (excluded from scoring for that transcript, same policy as H1/H2's ambiguous exclusion).
- **Q3**: `"yes"` if any turn (either participant) is labeled `echo`; `"no"` otherwise.

In [2]:
def speaker_labels(r):
    d = {}
    for sp, a in zip(r["speakers"], r["attribution"]):
        d.setdefault(sp, []).append(a["label"] if a else None)
    return d


def derive_answer_key(r):
    d = speaker_labels(r)
    sps = sorted(d.keys())
    strategic_sp = [sp for sp in sps if "strategic_persuasion" in d[sp]]
    q1 = strategic_sp[0] if len(strategic_sp) == 1 else "neither"

    ev_counts = {sp: d[sp].count("evidence_adoption") for sp in sps}
    zero_sps = [sp for sp, c in ev_counts.items() if c == 0]
    nonzero_sps = [sp for sp, c in ev_counts.items() if c >= 1]
    q2 = zero_sps[0] if len(zero_sps) == 1 and len(nonzero_sps) == 1 else "ambiguous"

    q3 = "yes" if any(label == "echo" for sp in sps for label in d[sp]) else "no"
    return {"q1_strategic_persuasion": q1, "q2_least_evidence_supported": q2, "q3_any_echo": q3}


strategic_ids = [r["transcript_id"] for r in results if "strategic_persuasion" in [a["label"] for a in r["attribution"] if a]]
print({"transcripts_with_real_strategic_persuasion_turn": strategic_ids})

clean_q2_ids = []
for r in results:
    key = derive_answer_key(r)
    if key["q2_least_evidence_supported"] != "ambiguous" and key["q3_any_echo"] == "yes":
        clean_q2_ids.append(r["transcript_id"])
print({"n_transcripts_with_unambiguous_q2_and_echo_present": len(clean_q2_ids)})

SELECTED_TRANSCRIPTS = strategic_ids[:2] + clean_q2_ids[:4]
answer_key_rows = []
for tid in SELECTED_TRANSCRIPTS:
    r = by_id[tid]
    key = derive_answer_key(r)
    answer_key_rows.append({"transcript_id": tid, "topic": r["topic"], **key})

answer_key = pd.DataFrame(answer_key_rows)
print(answer_key.to_string(index=False))


{'transcripts_with_real_strategic_persuasion_turn': ['125.0', '176.0', '303.0']}
{'n_transcripts_with_unambiguous_q2_and_echo_present': 66}
transcript_id                                                       topic q1_strategic_persuasion q2_least_evidence_supported q3_any_echo
        125.0   Should Governments Have the Right to Censor the Internet?               125.0_pro                   ambiguous         yes
        176.0 Is Government Surveillance Necessary for National Security?               176.0_con                   ambiguous         yes
        111.0               Should Students Have to Wear School Uniforms?                 neither                   111.0_con         yes
        121.0                             Should the Rich Pay More Taxes?                 neither                   121.0_con         yes
        127.0                     Should Felons Regain the Right to Vote?                 neither                   127.0_con         yes
        145.0               Shou

In [3]:
# Sanity check: is the workbench condition actually usable from this machine?
demo_data = root / "demo_site" / "data.json"
if not demo_data.is_file():
    print({
        "demo_site_data": "MISSING",
        "fix": f"run: python scripts/build_demo_site_data.py   (from {root})",
    })
else:
    demo_ids = {t["transcript_id"] for t in json.loads(demo_data.read_text(encoding="utf-8"))}
    missing = [t for t in SELECTED_TRANSCRIPTS if t not in demo_ids]
    print({
        "demo_site_data": "ok",
        "transcripts_in_demo": len(demo_ids),
        "selected_transcripts_missing_from_demo": missing,
        "static_site": str(root / "demo_site" / "index.html"),
        "streamlit": f"streamlit run app.py  (from {root}) -> 'Browse a real DebateGPT example'",
    })
    if missing:
        print("FIX REQUIRED before workbench sessions: regenerate data.json with scripts/build_demo_site_data.py")


{'demo_site_data': 'ok', 'transcripts_in_demo': 150, 'selected_transcripts_missing_from_demo': [], 'static_site': '/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/demo_site/index.html', 'streamlit': "streamlit run app.py  (from /home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer) -> 'Browse a real DebateGPT example'"}


## Counterbalanced condition assignment

Each of the 6 selected transcripts is assigned to exactly one participant under `workbench` and a *different* participant under `raw_transcript`. Each participant gets 4 transcripts total (2 per condition) and never sees the same transcript twice, so every transcript yields one real workbench judgment and one real raw-transcript judgment from different people.

In [4]:
# (transcript_id, workbench_participant, raw_transcript_participant); each transcript once per condition,
# each participant appears exactly twice per condition across the 6 rows -> 4 transcripts/participant.
ASSIGNMENT_PLAN = [
    (SELECTED_TRANSCRIPTS[0], "TS", "GS"),
    (SELECTED_TRANSCRIPTS[1], "TS", "RK"),
    (SELECTED_TRANSCRIPTS[2], "GS", "RK"),
    (SELECTED_TRANSCRIPTS[3], "GS", "TS"),
    (SELECTED_TRANSCRIPTS[4], "RK", "TS"),
    (SELECTED_TRANSCRIPTS[5], "RK", "GS"),
]

assignment_rows = []
for tid, wb, raw in ASSIGNMENT_PLAN:
    assignment_rows.append({"transcript_id": tid, "participant_id": wb, "condition": "workbench"})
    assignment_rows.append({"transcript_id": tid, "participant_id": raw, "condition": "raw_transcript"})
assignment = pd.DataFrame(assignment_rows)

per_participant_check = assignment.groupby(["participant_id", "condition"]).size().unstack(fill_value=0)
print(per_participant_check)
assert (per_participant_check == 2).all().all(), "each participant must get exactly 2 transcripts per condition"
assert assignment.groupby("transcript_id")["condition"].nunique().eq(2).all(), "each transcript must appear under both conditions"
assert assignment.groupby("transcript_id")["participant_id"].nunique().eq(2).all(), "each transcript must be judged by two different participants"
print({"status": "assignment_valid", "n_rows": len(assignment)})


condition       raw_transcript  workbench
participant_id                           
GS                           2          2
RK                           2          2
TS                           2          2
{'status': 'assignment_valid', 'n_rows': 12}


## Build real packets and blank answer sheets

For `raw_transcript` rows, the packet embeds the actual six real turns (speaker + text, in order) reconstructed from the system's own stored output -- no rhetorical tags, no stance chart, no attribution. For `workbench` rows, the packet points to the real demo surfaces and the transcript's topic/id so the participant can find and use it. Every row has blank `q1_answer`, `q2_answer`, `q3_answer`, and `time_to_answer_minutes` columns for the participant to fill in for real.

In [5]:
def raw_transcript_text(tid):
    r = by_id[tid]
    lines = [f"Topic: {r['topic']}", ""]
    for i, (sp, text) in enumerate(zip(r["speakers"], r["texts"])):
        side = sp.split("_")[-1]
        lines.append(f"Turn {i + 1} ({side}): {text}")
    return "\n\n".join(lines)


INSTRUCTIONS = (
    "For each transcript below, answer three questions about the debate:\n"
    "  Q1. Which participant (pro/con) was most influenced by strategic persuasion -- an abrupt shift to "
    "emotional/moral appeals right before a large stance change? Answer 'pro', 'con', or 'neither'.\n"
    "  Q2. Which participant's stance change was least supported by evidence -- i.e. they changed their "
    "position without ever citing a new, concrete fact or study? Answer 'pro', 'con', or \"can't tell\".\n"
    "  Q3. Did either participant clearly restate or mirror the other side's wording/framing at any point "
    "(echo behavior)? Answer 'yes' or 'no'.\n"
    "Please time yourself (in minutes) from when you start a transcript to when you finish all three answers "
    "for it, and record that in time_to_answer_minutes. Work independently; do not discuss answers with the "
    "other two participants until everyone has finished."
)

WORKBENCH_INSTRUCTIONS = (
    "WORKBENCH condition: open demo_site/index.html (or run the Streamlit app in 'browse real precomputed "
    "example' mode) and search for this transcript by its topic below. Use the rhetorical tag strips, the "
    "stance chart, and the Why? cards to help you answer."
)
RAW_INSTRUCTIONS = (
    "RAW-TRANSCRIPT condition: read only the plain transcript text embedded below. Do not open the workbench "
    "or demo site for this transcript."
)

user_study_dir = root / "data" / "user_study"
user_study_dir.mkdir(parents=True, exist_ok=True)

packets = {}
for index, participant in enumerate(["TS", "GS", "RK"]):
    rows = assignment[assignment["participant_id"] == participant].merge(answer_key[["transcript_id", "topic"]], on="transcript_id")
    packet_rows = []
    for _, row in rows.iterrows():
        tid = row["transcript_id"]
        condition = row["condition"]
        packet_rows.append({
            "participant_id": participant,
            "transcript_id": tid,
            "topic": row["topic"],
            "condition": condition,
            "material": WORKBENCH_INSTRUCTIONS if condition == "workbench" else RAW_INSTRUCTIONS + "\n\n" + raw_transcript_text(tid),
            "q1_answer": "",
            "q2_answer": "",
            "q3_answer": "",
            "time_to_answer_minutes": "",
        })
    packet_df = pd.DataFrame(packet_rows).sample(frac=1, random_state=42 + index).reset_index(drop=True)
    packet_path = user_study_dir / f"user_study_{participant}.csv"
    with packet_path.open("w", encoding="utf-8", newline="") as f:  # newline="": Windows-safe to_csv
        for line in INSTRUCTIONS.splitlines():
            f.write(f"# {line}\n")
        packet_df.to_csv(f, index=False)
    packets[participant] = packet_path
    print({"participant": participant, "n_transcripts": len(packet_df), "packet": str(packet_path)})


{'participant': 'TS', 'n_transcripts': 4, 'packet': '/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/data/user_study/user_study_TS.csv'}
{'participant': 'GS', 'n_transcripts': 4, 'packet': '/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/data/user_study/user_study_GS.csv'}
{'participant': 'RK', 'n_transcripts': 4, 'packet': '/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/data/user_study/user_study_RK.csv'}


## Save researcher-only files and the single fillable results sheet

`answer_key_master.csv` and `assignment_master.csv` are researcher-only (never shown to participants) so they can be used later to score returned answers. `results_fillable.csv` is the single sheet where the 12 real per-transcript judgments get typed in (per participant row by row). `results.csv` is intentionally **not** created here -- it only becomes real once the three participants actually complete the task and their real answers are scored by the final cell of this notebook.

In [6]:
answer_key_path = user_study_dir / "answer_key_master.csv"
assignment_path = user_study_dir / "assignment_master.csv"
answer_key.to_csv(answer_key_path, index=False)
assignment.to_csv(assignment_path, index=False)

# Single fillable sheet with all 12 judged rows (one row per participant x transcript x condition).
# If it already exists, any answers already filled in are preserved verbatim (never overwritten).
fillable_path = user_study_dir / "results_fillable.csv"
blank = assignment.merge(answer_key[["transcript_id", "topic"]], on="transcript_id", how="left")
blank["q1_answer"] = ""
blank["q2_answer"] = ""
blank["q3_answer"] = ""
blank["time_to_answer_minutes"] = ""
if fillable_path.is_file():
    try:
        # dtype=str + keep_default_na=False: pure string round-trip, immune to pandas
        # float64 coercion of all-NaN columns and id/string dtype drift across OSes.
        prev = pd.read_csv(fillable_path, comment="#", dtype=str, keep_default_na=False)
        prev = prev.drop_duplicates(subset=["participant_id", "transcript_id", "condition"], keep="last")
        val_cols = ["q1_answer", "q2_answer", "q3_answer", "time_to_answer_minutes"]
        prev_vals = prev[["participant_id", "transcript_id", "condition"] + val_cols]
        blank = blank.merge(prev_vals, on=["participant_id", "transcript_id", "condition"], how="left", suffixes=("", "_prev"))
        for c in val_cols:
            pv = blank[f"{c}_prev"]
            blank[c] = pv.where(pv.str.strip() != "", blank[c])
            blank.drop(columns=[f"{c}_prev"], inplace=True)
    except Exception as exc:  # unreadable previous sheet -> regenerate blank rather than crash
        print({"results_fillable": "previous sheet unreadable, regenerating blank", "error": repr(exc)})

blank = blank[[
    "participant_id", "transcript_id", "topic", "condition",
    "q1_answer", "q2_answer", "q3_answer", "time_to_answer_minutes",
]]
# newline="" is required on Windows: without it to_csv's lineterminator is
# double-translated and blank lines appear between every data row.
with fillable_path.open("w", encoding="utf-8", newline="") as f:
    f.write("# H3 real results - fill in the 4 blank columns below for each row, then save.\n")
    f.write("# q1_answer: who was most influenced by strategic persuasion -- pro / con / neither\n")
    f.write("# q2_answer: whose stance change was least supported by evidence -- pro / con / can't tell\n")
    f.write("# q3_answer: did either participant show echo behavior -- yes / no\n")
    f.write("# time_to_answer_minutes: real minutes spent on that transcript's 3 questions (a number, e.g. 6 or 4.5)\n")
    f.write("# Use each row's own material (see the corresponding user_study_<ID>.csv packet for the transcript text\n")
    f.write("# or workbench instructions). Do not fabricate/estimate answers -- leave blank if genuinely unanswered.\n")
    blank.to_csv(f, index=False)

results_path = user_study_dir / "results.csv"
print({
    "status": "dispatched_awaiting_real_completion",
    "selected_transcripts": SELECTED_TRANSCRIPTS,
    "answer_key": str(answer_key_path),
    "assignment": str(assignment_path),
    "participant_packets": {p: str(path) for p, path in packets.items()},
    "fillable_sheet": str(fillable_path),
    "results_csv_exists": results_path.is_file(),
    "next_step": (
        "TS, GS, and RK each complete their own rows for real (workbench rows via demo_site/index.html or "
        "`streamlit run app.py` -> 'Browse a real DebateGPT example'; raw rows via the embedded plain transcript), "
        "then the final cell below scores the returned answers into data/user_study/results.csv."
    ),
})


{'status': 'dispatched_awaiting_real_completion', 'selected_transcripts': ['125.0', '176.0', '111.0', '121.0', '127.0', '145.0'], 'answer_key': '/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/data/user_study/answer_key_master.csv', 'assignment': '/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/data/user_study/assignment_master.csv', 'participant_packets': {'TS': '/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/data/user_study/user_study_TS.csv', 'GS': '/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/data/user_study/user_study_GS.csv', 'RK': '/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/data/user_study/user_study_RK.csv'}, 'fillable_sheet': '/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/data/user_study/results_fillable.csv', 'results_csv_exists': False, 'next_step': "TS, GS, and RK each complete their own rows for real (workbench rows via demo_site/index.html or `streamlit run app.py` -> 'Browse a 

## AFTER TS, GS and RK return: score real answers into `results.csv` (fail-closed)

Type each participant's returned answers into `results_fillable.csv` (or paste them in from the returned `user_study_<ID>.csv` packets). Then run the cell below. It **refuses to write anything** unless all 12 rows have real answers for all three questions plus a real time value — incomplete rows are listed, never guessed.

Scoring rules (fixed in advance, mirroring the answer-key rules):
- **Q1**: correct if the answer matches the key (`<id>_pro` counts as `pro`, `<id>_con` as `con`, `neither` as `neither`).
- **Q2**: if the key is `ambiguous`, the question is excluded from that transcript's score (same policy as H1/H2 ambiguous exclusion).
- **Q3**: exact `yes` / `no` match.
- `accuracy` per row = correct / scorable questions; `time_to_answer` = the participant's own minutes.

Output columns match notebook 03's H3 gate exactly (`participant_id, condition, accuracy, time_to_answer`, plus `transcript_id` for auditability).

In [7]:
results_path = user_study_dir / "results.csv"
fillable_path = user_study_dir / "results_fillable.csv"

if not fillable_path.is_file():
    raise FileNotFoundError(f"run the packet cells first, then fill: {fillable_path}")

sheet = pd.read_csv(fillable_path, comment="#")
if sheet.duplicated(subset=["participant_id", "transcript_id", "condition"]).any():
    dup = sheet[sheet.duplicated(subset=["participant_id", "transcript_id", "condition"], keep=False)]
    raise ValueError(f"duplicate rows in {fillable_path}: {dup.to_dict('records')}")

expected = assignment[["participant_id", "transcript_id", "condition"]].astype(str)
sheet_keys = sheet[["participant_id", "transcript_id", "condition"]].astype(str)
merged = expected.merge(
    sheet_keys.assign(_row=range(len(sheet_keys))),
    how="left", on=["participant_id", "transcript_id", "condition"],
)
missing_rows = merged[merged["_row"].isna()]
if len(missing_rows):
    raise ValueError(
        f"results_fillable.csv is missing assignment rows: "
        f"{missing_rows.drop(columns='_row').to_dict('records')}"
    )
sheet = sheet.iloc[merged["_row"].astype(int)].reset_index(drop=True)


def norm(v):
    if pd.isna(v):
        return None
    s = str(v).strip().lower().strip("'\".")
    return s or None


def norm_side(v):
    s = str(v).strip().lower()
    if s.endswith("_pro"):
        return "pro"
    if s.endswith("_con"):
        return "con"
    return s


key_by_id = answer_key.assign(transcript_id=answer_key["transcript_id"].astype(str)).set_index("transcript_id")
problems, scored = [], []
for _, row in sheet.iterrows():
    key = key_by_id.loc[str(row["transcript_id"])]
    a1, a2, a3 = norm(row["q1_answer"]), norm(row["q2_answer"]), norm(row["q3_answer"])
    t = row["time_to_answer_minutes"]
    pid, tid, cond = row["participant_id"], row["transcript_id"], row["condition"]
    missing = [q for q, a in (("q1", a1), ("q2", a2), ("q3", a3)) if a is None]
    if pd.isna(t) or not str(t).strip():
        missing.append("time_to_answer_minutes")
    if missing:
        problems.append({"participant_id": pid, "transcript_id": tid, "condition": cond, "missing": missing})
        continue

    correct, scorable = 0, 0
    if norm_side(key["q1_strategic_persuasion"]) != "ambiguous":
        scorable += 1
        correct += int(a1 == norm_side(key["q1_strategic_persuasion"]))
    if norm_side(key["q2_least_evidence_supported"]) != "ambiguous":
        scorable += 1
        correct += int(a2 == norm_side(key["q2_least_evidence_supported"]))
    scorable += 1
    correct += int(a3 == norm_side(key["q3_any_echo"]))
    if scorable == 0:  # cannot happen with this answer key; kept fail-closed
        problems.append({"participant_id": pid, "transcript_id": tid, "condition": cond, "missing": ["all questions ambiguous"]})
        continue

    scored.append({
        "participant_id": pid,
        "transcript_id": tid,
        "condition": cond,
        "accuracy": correct / scorable,
        "time_to_answer": float(t),
    })

if problems:
    print({"status": "awaiting_real_answers", "n_complete_rows": len(scored), "incomplete_rows": problems})
    print(f"NOT writing {results_path}: fill every row for real first (answers are never guessed).")
else:
    results_df = pd.DataFrame(scored)[["participant_id", "transcript_id", "condition", "accuracy", "time_to_answer"]]
    results_df.to_csv(results_path, index=False)
    print({
        "status": "results_written",
        "results_csv": str(results_path),
        "n_rows": len(results_df),
        "n_participants": int(results_df["participant_id"].nunique()),
        "accuracy_by_condition": results_df.groupby("condition")["accuracy"].mean().to_dict(),
        "mean_time_by_condition": results_df.groupby("condition")["time_to_answer"].mean().to_dict(),
    })
    print("Now re-run the H3 cell in notebook 03 to compute the real H3 verdict from this file.")


{'status': 'results_written', 'results_csv': '/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/data/user_study/results.csv', 'n_rows': 12, 'n_participants': 3, 'accuracy_by_condition': {'raw_transcript': 0.5277777777777778, 'workbench': 0.6944444444444443}, 'mean_time_by_condition': {'raw_transcript': 2.2416666666666667, 'workbench': 2.5666666666666664}}
Now re-run the H3 cell in notebook 03 to compute the real H3 verdict from this file.
